# Aiscern Text Detector — Fine-Tuning
**Platform:** Google Colab (T4 GPU or CPU — CPU works)  
**Target:** `saghi776/aiscern-text-detector`  
**Base model:** `roberta-base`  
**Expected accuracy:** >90% on HC3 benchmark  
**Runtime:** ~30 minutes

### Before running:
1. Add `HF_TOKEN` in the 🔑 Secrets sidebar
2. Create HF repo: https://huggingface.co/new?owner=saghi776&name=aiscern-text-detector
3. Runtime → Change runtime type → T4 GPU (optional — CPU also works in ~45min)

In [ ]:
# CELL 1 — Install
!pip install -q transformers datasets accelerate huggingface_hub scikit-learn

In [ ]:
# CELL 2 — Authenticate
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)
print('✅ Authenticated')

In [ ]:
# CELL 3 — Configuration
BASE_MODEL  = 'roberta-base'
REPO_ID     = 'saghi776/aiscern-text-detector'
MAX_LENGTH  = 512
BATCH_SIZE  = 16
EPOCHS      = 3
LR          = 2e-5
SEED        = 42

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# CELL 4 — Load datasets
from datasets import load_dataset, concatenate_datasets, Dataset
import pandas as pd

records = []

# Dataset 1: HC3 — Human ChatGPT Comparison Corpus (gold standard)
# Contains: question, human_answers, chatgpt_answers
try:
    print('Loading HC3...')
    hc3 = load_dataset('Hello-SimpleAI/HC3', 'all', split='train')
    for row in hc3:
        for h in (row.get('human_answers') or []):
            if h and len(h.strip()) > 80:
                records.append({'text': h.strip()[:3000], 'label': 0})  # HUMAN
        for a in (row.get('chatgpt_answers') or []):
            if a and len(a.strip()) > 80:
                records.append({'text': a.strip()[:3000], 'label': 1})  # AI
    print(f'HC3: {len(records):,} records so far')
except Exception as e:
    print(f'HC3 skipped: {e}')

# Dataset 2: NicolaiSivesind/ChatGPT-Real
try:
    print('Loading ChatGPT-Real...')
    cr = load_dataset('NicolaiSivesind/ChatGPT-Real', split='train')
    for row in cr:
        text = row.get('text') or row.get('content') or ''
        lbl  = row.get('label') or row.get('source') or ''
        if text and len(text.strip()) > 80:
            lbl_lower = str(lbl).lower()
            is_ai = any(x in lbl_lower for x in ['gpt', 'chatgpt', 'ai', '1', 'generated'])
            records.append({'text': text.strip()[:3000], 'label': 1 if is_ai else 0})
    print(f'ChatGPT-Real: running total {len(records):,}')
except Exception as e:
    print(f'ChatGPT-Real skipped: {e}')

# Dataset 3: RAID benchmark (diverse AI generators)
try:
    print('Loading RAID...')
    raid = load_dataset('liamdugan/raid', split='train')
    for row in raid:
        text   = row.get('generation') or row.get('text') or ''
        source = row.get('model') or row.get('detector') or ''
        if text and len(text.strip()) > 80:
            is_ai = source and str(source).lower() not in ('human', 'none', '')
            records.append({'text': text.strip()[:3000], 'label': 1 if is_ai else 0})
    print(f'RAID: running total {len(records):,}')
except Exception as e:
    print(f'RAID skipped: {e}')

if len(records) < 1000:
    raise RuntimeError(f'Not enough data: only {len(records)} records. Check HF_TOKEN.')

print(f'\n✅ Total records: {len(records):,}')

In [ ]:
# CELL 5 — Balance and split
import pandas as pd

df = pd.DataFrame(records).sample(frac=1, random_state=SEED).reset_index(drop=True)
ai_count    = df.label.sum()
human_count = len(df) - ai_count
print(f'Raw: {len(df):,} total — AI: {ai_count:,}  Human: {human_count:,}')

# Balance classes
n = min(ai_count, human_count)
df_ai    = df[df.label == 1].head(n)
df_human = df[df.label == 0].head(n)
df_bal   = pd.concat([df_ai, df_human]).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f'Balanced: {len(df_bal):,} total — {n:,} AI + {n:,} Human')

# Train/val/test: 80/10/10
total    = len(df_bal)
train_df = df_bal[:int(total * 0.80)]
val_df   = df_bal[int(total * 0.80):int(total * 0.90)]
test_df  = df_bal[int(total * 0.90):]

print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

In [ ]:
# CELL 6 — Tokenize
from transformers import RobertaTokenizerFast
from datasets import Dataset

tokenizer = RobertaTokenizerFast.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
    )

# Convert to HF Datasets
train_ds = Dataset.from_pandas(train_df[['text', 'label']])
val_ds   = Dataset.from_pandas(val_df[['text', 'label']])
test_ds  = Dataset.from_pandas(test_df[['text', 'label']])

# Tokenize
train_ds = train_ds.map(tokenize, batched=True, desc='Tokenize train')
val_ds   = val_ds.map(tokenize, batched=True, desc='Tokenize val')
test_ds  = test_ds.map(tokenize, batched=True, desc='Tokenize test')

# Set format
cols = ['input_ids', 'attention_mask', 'label']
train_ds.set_format('torch', columns=cols)
val_ds.set_format('torch', columns=cols)
test_ds.set_format('torch', columns=cols)
print('✅ Tokenization complete')

In [ ]:
# CELL 7 — Load RoBERTa model
from transformers import RobertaForSequenceClassification

# CRITICAL id2label for hf-analyze.ts:
# /ai|generated|gpt|chatgpt/i → matches 'AI'
# /real|human|authentic/i      → matches 'HUMAN'
model = RobertaForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: 'HUMAN', 1: 'AI'},
    label2id={'HUMAN': 0, 'AI': 1},
)
print(f'Params: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M')

In [ ]:
# CELL 8 — Training
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'f1':       float(f1_score(labels, preds, average='binary')),
    }

training_args = TrainingArguments(
    output_dir                  = './aiscern-text-detector',
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate               = LR,
    warmup_steps                = 200,
    weight_decay                = 0.01,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    greater_is_better           = True,
    fp16                        = device == 'cuda',
    push_to_hub                 = True,
    hub_model_id                = REPO_ID,
    hub_token                   = HF_TOKEN,
    hub_strategy                = 'every_save',
    report_to                   = 'none',
    seed                        = SEED,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print('Starting training...')
trainer.train()
print('✅ Training complete')

In [ ]:
# CELL 9 — Evaluate
test_results = trainer.evaluate(test_ds)
print(f"\n{'='*40}")
print('TEST RESULTS')
print(f"{'='*40}")
print(f"Accuracy: {test_results['eval_accuracy']:.4f} ({test_results['eval_accuracy']*100:.1f}%)")
print(f"F1:       {test_results['eval_f1']:.4f}")
print(f"\nTarget (>90%): {'✅' if test_results['eval_accuracy'] >= 0.90 else '⚠️ '}")

In [ ]:
# CELL 10 — Push to HuggingFace
acc = test_results['eval_accuracy']
f1  = test_results['eval_f1']

trainer.push_to_hub(
    commit_message=f'RoBERTa AI text detector — acc={acc:.3f} f1={f1:.3f} | Aiscern text detector'
)
tokenizer.push_to_hub(REPO_ID, token=HF_TOKEN)
print(f'✅ Pushed to https://huggingface.co/{REPO_ID}')
print('\nTo wire into hf-analyze.ts, add to MODELS:')
print(f'  text_finetuned: "{REPO_ID}"')

In [ ]:
# CELL 11 — Quick inference test
import time
print('Waiting 30s for HF to load model...')
time.sleep(30)

from transformers import pipeline

pipe = pipeline('text-classification', model=REPO_ID, token=HF_TOKEN)

ai_text   = 'As an AI language model, I can provide comprehensive information on this topic. It is important to note that there are several key factors to consider when analyzing this multifaceted issue.'
real_text = 'Honestly I have no idea why this keeps happening lol. Tried restarting three times and still the same error. Anyone else seeing this?'

print('\n--- AI text (expected: AI > 0.7) ---')
r = pipe(ai_text)
print(f'  {r}')

print('\n--- Human text (expected: HUMAN > 0.7) ---')
r = pipe(real_text)
print(f'  {r}')

print('\n✅ Model working correctly')